<a href="https://colab.research.google.com/github/fahadahmad84-design/Quantum-Computing-Labs/blob/main/notebooks/Week_05_QC_M34099_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 Laboratory: Quantum Communication

[![View in GitHub](https://img.shields.io/badge/View%20in-GitHub-black?logo=github)](https://github.com/fahadahmad84-design/Quantum-Computing-Labs/blob/main/notebooks/Week_05_QC_M34099_Student.ipynb)

**M34099 Quantum Computing: Level 7 (2026/27)**

This laboratory contains three concise tasks:

1. create a Bell pair and observe correlated results;
2. demonstrate quantum teleportation and superdense coding; and
3. demonstrate BB84 and compare simulator results with IBM quantum hardware.

> **Saving your work:** Select **File > Save a copy in Drive** before editing. Never place an API key or CRN in a visible notebook cell.


## Set up Google Colab

Run the installation cell once. If Colab requests a restart, select **Runtime > Restart session**, then continue from the imports cell.


In [ ]:
# Install Qiskit, the Aer simulator, IBM Runtime, and drawing support.
%pip install -q "qiskit[visualization]" qiskit-aer qiskit-ibm-runtime pylatexenc


In [ ]:
# Import the Qiskit circuit and transpiler tools.
from qiskit import QuantumCircuit, transpile
# Import the local ideal simulator.
from qiskit_aer import AerSimulator
# Import the histogram plotting function.
from qiskit.visualization import plot_histogram

# Set the number of repeated measurements.
SHOTS = 1024
# Set a fixed seed for repeatable simulator results.
SEED = 34099
# Create the ideal simulator.
simulator = AerSimulator()
# Confirm that the setup is ready.
print("Setup complete.")


## Task 1: Create a Bell pair

An H gate creates superposition on qubit 0. A CNOT then entangles qubits 0 and 1. Measurement should produce only `00` and `11`, with approximately equal probabilities.


In [ ]:
# Create a circuit with two qubits and two classical bits.
bell_circuit = QuantumCircuit(2, 2)
# Put qubit 0 into an equal superposition.
bell_circuit.h(0)
# Entangle qubit 1 with qubit 0.
bell_circuit.cx(0, 1)
# Measure both qubits.
bell_circuit.measure([0, 1], [0, 1])
# Display the circuit.
display(bell_circuit.draw("mpl"))

# Convert the circuit for the simulator.
compiled_bell = transpile(bell_circuit, simulator)
# Run the circuit many times.
bell_result = simulator.run(compiled_bell, shots=SHOTS, seed_simulator=SEED).result()
# Extract the measurement counts.
bell_counts = bell_result.get_counts()
# Display the counts.
print("Bell-pair counts:", bell_counts)
# Display the histogram.
display(plot_histogram(bell_counts, title="Bell pair: correlated results"))


### Result

The outcomes are random, but they match. If the first qubit is measured as 0, the second is also 0. If the first is measured as 1, the second is also 1. Matching alone does not prove entanglement, but it shows the expected Z-basis correlation.


## Task 2A: Quantum teleportation

This circuit transfers the state $|1\rangle$ from qubit 0 to qubit 2. The controlled correction gates are the coherent equivalent of Alice sending two classical bits to Bob. Qubit 2 should be measured as 1.


In [ ]:
# Create three qubits and one classical output bit.
teleportation = QuantumCircuit(3, 1)
# Prepare the state one on qubit 0.
teleportation.x(0)
# Create a Bell pair between qubits 1 and 2.
teleportation.h(1)
# Complete the Bell pair.
teleportation.cx(1, 2)
# Start Alice's Bell-state operation.
teleportation.cx(0, 1)
# Complete Alice's Bell-state operation.
teleportation.h(0)
# Apply Bob's X correction coherently.
teleportation.cx(1, 2)
# Apply Bob's Z correction coherently.
teleportation.cz(0, 2)
# Measure Bob's qubit.
teleportation.measure(2, 0)
# Display the circuit.
display(teleportation.draw("mpl"))

# Convert the circuit for the simulator.
compiled_teleportation = transpile(teleportation, simulator)
# Run the teleportation circuit.
teleportation_result = simulator.run(compiled_teleportation, shots=SHOTS, seed_simulator=SEED).result()
# Extract Bob's counts.
teleportation_counts = teleportation_result.get_counts()
# Display Bob's result.
print("Bob's teleported state:", teleportation_counts)
# Display the histogram.
display(plot_histogram(teleportation_counts, title="Teleportation of |1>"))


### Result

Bob's qubit is measured as 1, matching the state initially prepared on qubit 0. The original state is transferred rather than copied.


## Task 2B: Superdense coding

Alice and Bob share one Bell pair. Alice encodes the classical message `10` by applying Z to her qubit. Bob decodes the pair and should recover `10`.


In [ ]:
# Create two qubits and two classical bits.
superdense = QuantumCircuit(2, 2)
# Create superposition on Alice's qubit.
superdense.h(0)
# Create the shared Bell pair.
superdense.cx(0, 1)
# Apply Z to encode the first message bit as one.
superdense.z(0)
# Apply CNOT as the first decoding operation.
superdense.cx(0, 1)
# Apply H as the second decoding operation.
superdense.h(0)
# Store qubit 0 as the left message bit.
superdense.measure(0, 1)
# Store qubit 1 as the right message bit.
superdense.measure(1, 0)
# Display the circuit.
display(superdense.draw("mpl"))

# Convert the circuit for the simulator.
compiled_superdense = transpile(superdense, simulator)
# Run the superdense-coding circuit.
superdense_result = simulator.run(compiled_superdense, shots=SHOTS, seed_simulator=SEED).result()
# Extract the decoded message counts.
superdense_counts = superdense_result.get_counts()
# Display Bob's decoded message.
print("Decoded message:", superdense_counts)
# Display the histogram.
display(plot_histogram(superdense_counts, title="Superdense coding: message 10"))


### Result

Bob recovers `10`. The protocol uses one transmitted qubit and one previously shared Bell pair to communicate two classical bits.


## Task 3: BB84 on a simulator

BB84 uses the states $|0\rangle$, $|1\rangle$, $|+\rangle$ and $|-\rangle$. The three circuits below demonstrate a Z-basis match, an X-basis match and a basis mismatch.


In [ ]:
# Create a function for a simple BB84 circuit.
def make_bb84_circuit(bit, alice_basis, bob_basis):
    # Create one qubit and one classical bit.
    circuit = QuantumCircuit(1, 1)
    # Apply X when Alice encodes bit one.
    if bit == 1:
        # Change the initial state from zero to one.
        circuit.x(0)
    # Apply H when Alice prepares an X-basis state.
    if alice_basis == "X":
        # Convert zero or one into plus or minus.
        circuit.h(0)
    # Apply H when Bob measures in the X basis.
    if bob_basis == "X":
        # Convert the X basis into the standard measurement basis.
        circuit.h(0)
    # Measure the qubit.
    circuit.measure(0, 0)
    # Return the completed circuit.
    return circuit

# Prepare |0> and measure it in the matching Z basis.
bb84_z_match = make_bb84_circuit(0, "Z", "Z")
# Prepare |+> and measure it in the matching X basis.
bb84_x_match = make_bb84_circuit(0, "X", "X")
# Prepare |+> and measure it in the different Z basis.
bb84_mismatch = make_bb84_circuit(0, "X", "Z")
# Store the circuits in one list.
bb84_circuits = [bb84_z_match, bb84_x_match, bb84_mismatch]
# Store clear labels for the circuits.
bb84_labels = ["Z match", "X match", "Different bases"]

# Convert the circuits for the simulator.
compiled_bb84 = [transpile(circuit, simulator) for circuit in bb84_circuits]

# Run the circuits on the local ideal simulator.
simulator_bb84_counts = {}
for label, circuit in zip(bb84_labels, compiled_bb84):
    result = simulator.run(circuit, shots=SHOTS, seed_simulator=SEED).result()
    simulator_bb84_counts[label] = result.get_counts()

# Display the results.
print("Simulator results:", simulator_bb84_counts)

# Display the histogram using a list of counts and the labels as the legend.
display(plot_histogram(
    list(simulator_bb84_counts.values()),
    legend=list(simulator_bb84_counts.keys()),
    title="BB84: simulator results"
))

### Simulator result

The two matching-basis circuits return 0 with certainty. The different-basis circuit returns approximately equal numbers of 0 and 1. An interceptor who measures in the wrong basis can therefore disturb the transmitted information.


## Optional: Run Task 3 on IBM quantum hardware

Run this section only after your tutor confirms that IBM Quantum Classroom access is ready. The same three BB84 circuits are used so the simulator and hardware results can be compared directly.

- Keep `RUN_ON_REAL_HARDWARE = False` until access is ready.
- Enter your API key and classroom CRN only through the hidden prompts.
- Never save credentials in the notebook, Google Drive, Moodle or GitHub.


In [ ]:
# Keep real-hardware execution disabled until access is ready.
RUN_ON_REAL_HARDWARE = False
# Display the current hardware setting.
print("Real-hardware execution enabled:", RUN_ON_REAL_HARDWARE)


In [ ]:
# Continue only when real-hardware execution is enabled.
if RUN_ON_REAL_HARDWARE:
    # Import secure hidden-input support.
    from getpass import getpass
    # Import the IBM Quantum Runtime connection.
    from qiskit_ibm_runtime import QiskitRuntimeService
    # Import the device-aware circuit pass manager.
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    # Import the IBM Runtime Sampler interface.
    from qiskit_ibm_runtime import SamplerV2

    # Connect only once during the current Colab session.
    if "ibm_service" not in globals():
        # Request the API key without displaying it.
        ibm_api_key = getpass("Enter your personal IBM Cloud API key: ")
        # Request the classroom CRN without displaying it.
        classroom_crn = getpass("Enter the classroom instance CRN: ")

        # Connect to the classroom instance.
        ibm_service = QiskitRuntimeService(
            channel="ibm_quantum_platform",
            token=ibm_api_key,
            instance=classroom_crn,
        )

        # Remove the plaintext credential variables.
        del ibm_api_key, classroom_crn

    # Select an operational quantum computer with a short queue.
    backend = ibm_service.least_busy(
        operational=True,
        simulator=False,
    )

    # Display the selected device.
    print("Selected backend:", backend.name)

    # Create a pass manager for the selected device.
    pass_manager = generate_preset_pass_manager(
        backend=backend,
        optimization_level=1,
    )

    # Convert the same BB84 circuits for the real device.
    hardware_circuits = [
        pass_manager.run(circuit)
        for circuit in bb84_circuits
    ]

    # Create the Runtime Sampler.
    sampler = SamplerV2(mode=backend)

    # Submit all three circuits in one job.
    hardware_job = sampler.run(
        hardware_circuits,
        shots=SHOTS,
    )

    # Display the job identifier.
    print("Job ID:", hardware_job.job_id())

    # Wait for the job and collect the results.
    hardware_result = hardware_job.result()

    # Create an empty dictionary for the hardware counts.
    hardware_bb84_counts = {}

    # Extract the result for each circuit.
    for index, label in enumerate(bb84_labels):
        # Read the counts from the classical register.
        counts = hardware_result[index].data.c.get_counts()
        # Convert the Counts object into a standard dictionary.
        hardware_bb84_counts[label] = dict(counts)

    # Display all hardware counts.
    print("Hardware results:", hardware_bb84_counts)

    # Compare each circuit separately.
    for label in bb84_labels:
        # Read the simulator counts for this circuit.
        simulator_counts = dict(simulator_bb84_counts[label])
        # Read the hardware counts for this circuit.
        device_counts = hardware_bb84_counts[label]

        # Display a simulator and hardware comparison.
        display(plot_histogram(
            [simulator_counts, device_counts],
            legend=["Ideal simulator", f"IBM {backend.name}"],
            title=f"BB84 comparison: {label}",
        ))

# Skip the hardware job when the safety switch is off.
else:
    # Confirm that no hardware job was submitted.
    print("No hardware job submitted.")

## References

- IBM Quantum Learning, *Entanglement in action*: https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/entanglement-in-action
- IBM Quantum Learning, *Quantum key distribution*: https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/quantum-key-distribution
- IBM Quantum Documentation, *Run jobs with Qiskit Runtime*: https://quantum.cloud.ibm.com/docs/guides/run-jobs
- Qiskit Aer documentation: https://qiskit.github.io/qiskit-aer/

Accessed 19 October 2026.
